# 5. Python for Data Work

AI가 만든 코드의 이름·반환값·mutable state·함수 경계·model interface를 읽고 수정합니다.

핵심 셀은 유료 API, GPU, 외부 데이터 다운로드 없이 실행합니다. NumPy·pandas가 필요한 회차는 해당 패키지를 사용합니다. 수업용 고정 입력이며 실제 연구 성능을 주장하지 않습니다.

## 1. 같은 list를 가리키는 두 이름

결과를 먼저 예측하고 실행하세요. shallow copy가 내부 list까지 복사하는지 추가로 확인합니다.

In [1]:
scores = [.2,.5,.8]
alias = scores
alias.append(.95)
assert scores == [.2,.5,.8,.95]
independent = scores.copy()
independent.append(.99)
print(scores,independent)
nested = [[1],[2]]
shallow = nested.copy()
shallow[0].append(9)
assert nested[0] == [1,9]
print("Nested object is shared:",nested)

[0.2, 0.5, 0.8, 0.95] [0.2, 0.5, 0.8, 0.95, 0.99]
Nested object is shared: [[1, 9], [2]]


## 2. mutable default 재현과 수정

In [2]:
def bad_add(x, history=[]):
    history.append(x)
    return history
first = list(bad_add(.2))
second = list(bad_add(.8))
print(first,second)
assert first == [.2] and second == [.2,.8]
def add(x, history=None):
    values = [] if history is None else list(history)
    return values+[x]
assert add(.2)==[.2] and add(.8)==[.8]

[0.2] [0.2, 0.8]


## 3. 명시적 의존성과 예외

함수가 notebook의 숨은 전역 threshold에 의존하지 않게 합니다.

In [3]:
import math
def classify(scores, threshold=.5):
    if not math.isfinite(threshold) or not 0<=threshold<=1:
        raise ValueError("invalid threshold")
    values=list(scores)
    if any(not math.isfinite(x) or not 0<=x<=1 for x in values):
        raise ValueError("invalid score")
    return [int(x>=threshold) for x in values]
assert classify([.2,.5,.8],.7)==[0,0,1]
print(classify([.2,.5,.8],.7))

[0, 0, 1]


## 4. generator와 context manager

마지막 partial batch를 버리지 않고, scope를 벗어나면 자원을 닫습니다.

In [4]:
from io import StringIO
def batches(values,size):
    if size<=0: raise ValueError("positive batch size required")
    for start in range(0,len(values),size):
        yield values[start:start+size]
assert list(batches([1,2,3,4,5],2))==[[1,2],[3,4],[5]]
with StringIO("0.2\n0.5\n0.8\n") as handle:
    parsed=[float(line) for line in handle]
assert handle.closed
print(list(batches(parsed,2)))

[[0.2, 0.5], [0.8]]


## 5. Stateful Model Interface

class와 instance를 구분합니다. 빈 입력으로 fit할 때는 검증 가능한 실패를 내야 합니다.

In [5]:
class MeanModel:
    def fit(self,y):
        if not y: raise ValueError("empty training target")
        self.mean_=sum(y)/len(y)
        return self
    def predict(self,n):
        if not hasattr(self,"mean_"): raise RuntimeError("fit before predict")
        return [self.mean_]*n
a=MeanModel().fit([1,2,3])
b=MeanModel().fit([6,8])
assert a.predict(2)==[2.,2.] and b.predict(1)==[7.]
print(a.predict(2),b.predict(1))

[2.0, 2.0] [7.0]


## 6. 과제와 해답

- 두 model의 state가 공유되지 않는지 검사하세요.
- fit 전 predict와 빈 target fit이 명시적으로 실패하는지 검사하세요.

In [6]:
assert a.mean_==2 and b.mean_==7
for operation,error in [(lambda: MeanModel().predict(2),RuntimeError),(lambda: MeanModel().fit([]),ValueError)]:
    try: operation()
    except error as ex: print(type(ex).__name__,str(ex))
    else: raise AssertionError("expected failure did not occur")

RuntimeError fit before predict
ValueError empty training target


## 7. 참고: Protocol과 Dataclass

`__len__`, `__getitem__`은 익숙한 표기를 interface로 연결합니다. type hint와 dataclass는 런타임 유효성 검사를 자동으로 보장하지 않습니다.

In [7]:
from dataclasses import dataclass
@dataclass
class Config:
    threshold: float=.5
class ScoreData:
    def __init__(self,values): self.values=list(values)
    def __len__(self): return len(self.values)
    def __getitem__(self,index): return self.values[index]
data=ScoreData([.2,.5,.8])
assert len(data)==3 and data[1]==.5
print(Config(),len(data),data[1])

Config(threshold=0.5) 3 0.5


## 8. 참고: Decorator와 공통 Interface

wrapper는 원래 함수를 호출하는 전후에 동작을 더합니다. 복잡한 decorator factory, MRO, property 상세는 이 실습의 필수 조건이 아닙니다.

In [8]:
from functools import wraps
def counted(fn):
    @wraps(fn)
    def wrapper(*args,**kwargs):
        wrapper.calls+=1
        return fn(*args,**kwargs)
    wrapper.calls=0
    return wrapper
@counted
def square(x): return x*x
assert square(3)==9 and square.calls==1
print(square.__name__,square.calls)

square 1
